In [2]:
"""
tfidf_lstm.py
=============
Sentiment Analysis  —  TF-IDF Vocabulary  +  Bidirectional LSTM  (PyTorch)

Pipeline:
  1.  Load cleaned data
  2.  Build TF-IDF vocabulary  →  word-to-index mapping
  3.  Encode sentences as padded integer sequences
  4.  PyTorch Dataset & DataLoader
  5.  Bidirectional LSTM model
  6.  Train with Early Stopping + LR Scheduler
  7.  Evaluate  →  Accuracy, F1, Classification Report, Confusion Matrix
  8.  MLflow experiment tracking
  9.  Save best model
 10.  Plots  →  Training curves, Confusion matrix, Per-class F1
"""

import os
import re
import time
import logging
import warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.pytorch

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)


# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # Data
    "data_path"    : "data_cleaned.csv",
    "output_dir"   : "outputs",
    "save_dir"     : "saved_models",

    # Vocabulary
    "max_features" : 6000,      # TF-IDF top-N words
    "max_seq_len"  : 40,         # pad / truncate length

    # Model
    "embed_dim"    : 64,        # embedding size
    "hidden_dim"   : 128,        # LSTM hidden units
    "num_layers"   : 1,          # stacked LSTM layers
    "dropout"      : 0.3,        # dropout rate
    "num_classes"  : 3,

    # Training
    "batch_size"   : 128,
    "epochs"       : 10,
    "lr"           : 1e-3,
    "weight_decay" : 1e-4,
    "patience"     : 3,          # early stopping
    "grad_clip"    : 1.0,

    # Split
    "test_size"    : 0.20,
    "val_size"     : 0.10,
    "seed"         : 42,

    # Device
    "device"       : "cuda" if torch.cuda.is_available() else "cpu",

    # MLflow
    "mlflow_exp"   : "Sentiment_TFIDF_LSTM",
}

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
CLASS_NAMES = ["negative", "neutral", "positive"]

os.makedirs(CFG["save_dir"],  exist_ok=True)
os.makedirs(CFG["output_dir"], exist_ok=True)

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])


# ─────────────────────────────────────────────────────────────────────────────
# 1.  Load Data
# ─────────────────────────────────────────────────────────────────────────────
def load_data() -> pd.DataFrame:
    df = pd.read_csv(CFG["data_path"])
    df = df[["Cleaned_Sentence", "Sentiment"]].dropna()
    df["label"] = df["Sentiment"].map(LABEL2ID)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    df["label"] = df["label"].astype(int)
    logger.info(f"Loaded  →  {len(df):,} rows")
    logger.info(f"Label distribution:\n{df['Sentiment'].value_counts().to_string()}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Build TF-IDF Vocabulary
# ─────────────────────────────────────────────────────────────────────────────
def build_vocab(texts):
    """
    Fit TF-IDF on training texts to get top-N vocabulary.
    Returns:
        vocab  : {word: index}   (0 = PAD, last = UNK)
        tfidf  : fitted TfidfVectorizer
    """
    tfidf = TfidfVectorizer(
        max_features = CFG["max_features"],
        ngram_range  = (1, 1),
        min_df       = 2,
        sublinear_tf = True,
    )
    tfidf.fit(texts)

    vocab = {word: idx + 1                              # 1-indexed
             for idx, word in enumerate(tfidf.get_feature_names_out())}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = len(vocab)

    logger.info(f"Vocabulary built  →  {len(vocab):,} tokens")
    return vocab, tfidf


# ─────────────────────────────────────────────────────────────────────────────
# 3.  Sentence Encoding
# ─────────────────────────────────────────────────────────────────────────────
def encode_sentence(text: str, vocab: dict, max_len: int) -> list:
    tokens = str(text).lower().split()
    ids    = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    ids    = ids[:max_len]                               # truncate
    ids   += [0] * (max_len - len(ids))                 # pad
    return ids


def encode_dataset(texts, vocab: dict, max_len: int) -> np.ndarray:
    return np.array([encode_sentence(t, vocab, max_len) for t in texts])


# ─────────────────────────────────────────────────────────────────────────────
# 4.  Dataset & DataLoader
# ─────────────────────────────────────────────────────────────────────────────
class SentimentDataset(Dataset):
    def __init__(self, sequences: np.ndarray, labels: np.ndarray):
        self.X = torch.tensor(sequences, dtype=torch.long)
        self.y = torch.tensor(labels,    dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def get_dataloaders(df: pd.DataFrame, vocab: dict):
    texts  = df["Cleaned_Sentence"].values
    labels = df["label"].values

    # Train / temp
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        texts, labels,
        test_size  = CFG["test_size"] + CFG["val_size"],
        stratify   = labels,
        random_state = CFG["seed"],
    )
    # Val / test
    val_ratio = CFG["val_size"] / (CFG["test_size"] + CFG["val_size"])
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp,
        test_size    = 1 - val_ratio,
        stratify     = y_tmp,
        random_state = CFG["seed"],
    )

    X_tr_enc  = encode_dataset(X_tr,  vocab, CFG["max_seq_len"])
    X_val_enc = encode_dataset(X_val, vocab, CFG["max_seq_len"])
    X_te_enc  = encode_dataset(X_te,  vocab, CFG["max_seq_len"])

    train_dl = DataLoader(SentimentDataset(X_tr_enc,  y_tr),
                          batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=0)
    val_dl   = DataLoader(SentimentDataset(X_val_enc, y_val),
                          batch_size=CFG["batch_size"], num_workers=0)
    test_dl  = DataLoader(SentimentDataset(X_te_enc,  y_te),
                          batch_size=CFG["batch_size"], num_workers=0)

    logger.info(
        f"Split  →  Train: {len(X_tr):,}  |  Val: {len(X_val):,}  |  Test: {len(X_te):,}"
    )
    return train_dl, val_dl, test_dl, y_te


# ─────────────────────────────────────────────────────────────────────────────
# 5.  Bidirectional LSTM Model
# ─────────────────────────────────────────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    """
    Architecture:
        Embedding  →  Dropout
        →  Bidirectional LSTM  (num_layers stacked)
        →  Concat last forward + backward hidden state
        →  LayerNorm
        →  Dropout
        →  Fully Connected (hidden_dim*2  →  num_classes)
    """
    def __init__(self, vocab_size: int):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim  = CFG["embed_dim"],
            padding_idx    = 0,
        )
        self.embed_dropout = nn.Dropout(CFG["dropout"])

        self.lstm = nn.LSTM(
            input_size    = CFG["embed_dim"],
            hidden_size   = CFG["hidden_dim"],
            num_layers    = CFG["num_layers"],
            batch_first   = True,
            dropout       = CFG["dropout"] if CFG["num_layers"] > 1 else 0.0,
            bidirectional = True,
        )

        self.layer_norm = nn.LayerNorm(CFG["hidden_dim"] * 2)
        self.dropout    = nn.Dropout(CFG["dropout"])

        self.classifier = nn.Sequential(
            nn.Linear(CFG["hidden_dim"] * 2, CFG["hidden_dim"]),
            nn.ReLU(),
            nn.Dropout(CFG["dropout"]),
            nn.Linear(CFG["hidden_dim"], CFG["num_classes"]),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len)
        emb = self.embed_dropout(self.embedding(x))          # (B, L, E)

        _, (hn, _) = self.lstm(emb)
        # hn shape: (num_layers * 2, B, hidden_dim)
        # take the last layer's forward (hn[-2]) and backward (hn[-1])
        hidden = torch.cat([hn[-2], hn[-1]], dim=1)          # (B, H*2)

        hidden = self.layer_norm(hidden)
        hidden = self.dropout(hidden)
        return self.classifier(hidden)                        # (B, num_classes)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ─────────────────────────────────────────────────────────────────────────────
# 6.  Train / Eval Steps
# ─────────────────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += len(y)

    return total_loss / total, correct / total


def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for X, y in loader:
            X, y   = X.to(device), y.to(device)
            logits = model(X)
            loss   = criterion(logits, y)

            total_loss += loss.item() * len(y)
            preds       = logits.argmax(1)
            correct    += (preds == y).sum().item()
            total      += len(y)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    return total_loss / total, correct / total, all_preds, all_labels


# ─────────────────────────────────────────────────────────────────────────────
# 7.  Full Training Loop
# ─────────────────────────────────────────────────────────────────────────────
def train_model(model, train_dl, val_dl, device):
    logger.info(f"Parameters  →  {model.count_parameters():,}")
    logger.info(f"Device      →  {device}")

    # Weighted loss for class imbalance  (neg=590, neu=2874, pos=1843)
    counts  = torch.tensor([590.0, 2874.0, 1843.0])
    weights = (1.0 / counts)
    weights = (weights / weights.sum()).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = Adam(
        model.parameters(),
        lr           = CFG["lr"],
        weight_decay = CFG["weight_decay"],
    )
    scheduler = ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc" : [], "val_acc" : [],
    }

    best_val_loss  = float("inf")
    best_weights   = None
    patience_count = 0

    logger.info(f"\n{'='*60}")
    logger.info("  Starting Training  —  Bidirectional LSTM")
    logger.info(f"{'='*60}")

    for epoch in range(1, CFG["epochs"] + 1):
        t0 = time.time()

        tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer, criterion, device)
        vl_loss, vl_acc, _, _ = eval_one_epoch(model, val_dl, criterion, device)

        scheduler.step(vl_loss)
        elapsed = time.time() - t0

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        logger.info(
            f"Epoch {epoch:02d}/{CFG['epochs']}  |  "
            f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}  |  "
            f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}  |  "
            f"{elapsed:.1f}s"
        )

        # Early stopping
        if vl_loss < best_val_loss - 1e-4:
            best_val_loss  = vl_loss
            best_weights   = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= CFG["patience"]:
                logger.info(f"Early stopping triggered at epoch {epoch}")
                break

    # Restore best weights
    model.load_state_dict(best_weights)

    # Save model
    save_path = os.path.join(CFG["save_dir"], "bilstm_sentiment.pt")
    torch.save({
        "model_state_dict": model.state_dict(),
        "config"          : CFG,
        "label2id"        : LABEL2ID,
    }, save_path)
    logger.info(f"Best model saved  →  {save_path}")

    return model, history


# ─────────────────────────────────────────────────────────────────────────────
# 8.  Evaluate on Test Set
# ─────────────────────────────────────────────────────────────────────────────
def evaluate(model, test_dl, device) -> dict:
    criterion          = nn.CrossEntropyLoss()
    _, acc, preds, labels = eval_one_epoch(model, test_dl, criterion, device)

    f1_mac = f1_score(labels, preds, average="macro")
    f1_wei = f1_score(labels, preds, average="weighted")
    report = classification_report(labels, preds,
                                   target_names=CLASS_NAMES, digits=4)
    cm     = confusion_matrix(labels, preds)

    print(f"\n{'='*60}")
    print("  Bidirectional LSTM  —  Test Set Results")
    print(f"{'='*60}")
    print(f"  Accuracy     :  {acc:.4f}")
    print(f"  F1-Macro     :  {f1_mac:.4f}")
    print(f"  F1-Weighted  :  {f1_wei:.4f}")
    print(f"\n{report}")

    return {
        "accuracy"    : acc,
        "f1_macro"    : f1_mac,
        "f1_weighted" : f1_wei,
        "preds"       : preds,
        "labels"      : labels,
        "cm"          : cm,
    }


# ─────────────────────────────────────────────────────────────────────────────
# 9.  Plots
# ─────────────────────────────────────────────────────────────────────────────
BG    = "#0f1117"
PANEL = "#1a1d27"
BLUE  = "#3498db"
GREEN = "#2ecc71"
RED   = "#e74c3c"

def save_training_curves(history: dict):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)

    for ax, (tr_key, vl_key), title in zip(
        axes,
        [("train_loss","val_loss"), ("train_acc","val_acc")],
        ["Loss Curve", "Accuracy Curve"],
    ):
        ax.plot(history[tr_key], color=BLUE,  linewidth=2, label="Train")
        ax.plot(history[vl_key], color=GREEN, linewidth=2, label="Validation",
                linestyle="--")
        ax.set_facecolor(PANEL)
        ax.set_title(title, color="white", fontsize=13, fontweight="bold")
        ax.tick_params(colors="#aaa")
        ax.spines[["top","right"]].set_visible(False)
        ax.spines[["left","bottom"]].set_color("#333")
        ax.legend(facecolor=PANEL, labelcolor="white", fontsize=10)
        ax.set_xlabel("Epoch", color="#aaa")

    fig.suptitle("BiLSTM  —  Training Curves",
                 color="white", fontsize=16, fontweight="bold")
    plt.tight_layout()
    out = os.path.join(CFG["output_dir"], "lstm_training_curves.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    logger.info(f"Training curves saved  →  {out}")
    return out


def save_confusion_matrix(cm: np.ndarray):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=BG)
    ax.set_facecolor(PANEL)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(CLASS_NAMES, color="#aaa", fontsize=10)
    ax.set_yticklabels(CLASS_NAMES, color="#aaa", fontsize=10)
    ax.set_xlabel("Predicted", color="#aaa", fontsize=11)
    ax.set_ylabel("Actual",    color="#aaa", fontsize=11)
    ax.set_title("BiLSTM  —  Confusion Matrix",
                 color="white", fontsize=13, fontweight="bold")
    for i in range(3):
        for j in range(3):
            ax.text(j, i,
                    f"{cm[i,j]}\n({cm_norm[i,j]:.2f})",
                    ha="center", va="center", fontsize=10,
                    color="white" if cm_norm[i,j] > 0.5 else "#222")
    plt.tight_layout()
    out = os.path.join(CFG["output_dir"], "lstm_confusion_matrix.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    logger.info(f"Confusion matrix saved  →  {out}")
    return out


def save_per_class_f1(preds, labels):
    report = classification_report(labels, preds,
                                   target_names=CLASS_NAMES,
                                   output_dict=True)
    f1s    = [report[c]["f1-score"] for c in CLASS_NAMES]
    colors = [RED, BLUE, GREEN]

    fig, ax = plt.subplots(figsize=(7, 4), facecolor=BG)
    ax.set_facecolor(PANEL)
    bars = ax.bar(CLASS_NAMES, f1s, color=colors, edgecolor="none", width=0.5)
    for bar, v in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f"{v:.4f}", ha="center", color="white",
                fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("F1-Score", color="#aaa")
    ax.set_title("BiLSTM  —  Per-Class F1 Score",
                 color="white", fontsize=13, fontweight="bold")
    ax.tick_params(colors="#aaa")
    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_color("#333")
    plt.tight_layout()
    out = os.path.join(CFG["output_dir"], "lstm_per_class_f1.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    logger.info(f"Per-class F1 chart saved  →  {out}")
    return out


# ─────────────────────────────────────────────────────────────────────────────
# 10.  MLflow Logging
# ─────────────────────────────────────────────────────────────────────────────
def log_to_mlflow(results: dict, history: dict, artifacts: list):
    mlflow.set_experiment(CFG["mlflow_exp"])

    with mlflow.start_run(run_name="BiLSTM"):
        # Parameters
        mlflow.log_params({
            "model"        : "Bidirectional LSTM",
            "embed_dim"    : CFG["embed_dim"],
            "hidden_dim"   : CFG["hidden_dim"],
            "num_layers"   : CFG["num_layers"],
            "dropout"      : CFG["dropout"],
            "batch_size"   : CFG["batch_size"],
            "lr"           : CFG["lr"],
            "max_seq_len"  : CFG["max_seq_len"],
            "max_features" : CFG["max_features"],
            "optimizer"    : "Adam",
            "loss"         : "WeightedCrossEntropy",
        })

        # Final test metrics
        mlflow.log_metrics({
            "test_accuracy"   : results["accuracy"],
            "test_f1_macro"   : results["f1_macro"],
            "test_f1_weighted": results["f1_weighted"],
        })

        # Per-epoch metrics
        for epoch, (tl, vl, ta, va) in enumerate(zip(
            history["train_loss"], history["val_loss"],
            history["train_acc"],  history["val_acc"]
        ), 1):
            mlflow.log_metrics({
                "train_loss": tl, "val_loss": vl,
                "train_acc" : ta, "val_acc" : va,
            }, step=epoch)

        # Artifacts (plots + model)
        for path in artifacts:
            if os.path.exists(path):
                mlflow.log_artifact(path)

    logger.info("MLflow run logged  →  BiLSTM")


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────
def main():
    device = torch.device(CFG["device"])

    # 1. Load
    df = load_data()

    # 2. Vocabulary
    vocab, _ = build_vocab(df["Cleaned_Sentence"])

    # 3. DataLoaders
    train_dl, val_dl, test_dl, y_te = get_dataloaders(df, vocab)

    # 4. Model
    model = BiLSTMClassifier(vocab_size=len(vocab)).to(device)
    logger.info(f"Model parameters  →  {model.count_parameters():,}")

    # 5. Train
    model, history = train_model(model, train_dl, val_dl, device)

    # 6. Evaluate
    results = evaluate(model, test_dl, device)

    # 7. Plots
    p1 = save_training_curves(history)
    p2 = save_confusion_matrix(results["cm"])
    p3 = save_per_class_f1(results["preds"], results["labels"])

    # 8. MLflow
    model_path = os.path.join(CFG["save_dir"], "bilstm_sentiment.pt")
    log_to_mlflow(results, history, [p1, p2, p3, model_path])

    # 9. Final summary
    print(f"\n{'='*60}")
    print("  FINAL SUMMARY")
    print(f"{'='*60}")
    print(f"  Accuracy     :  {results['accuracy']:.4f}")
    print(f"  F1-Macro     :  {results['f1_macro']:.4f}")
    print(f"  F1-Weighted  :  {results['f1_weighted']:.4f}")
    print(f"  Model saved  :  {model_path}")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    main()


  Bidirectional LSTM  —  Test Set Results
  Accuracy     :  0.5823
  F1-Macro     :  0.5229
  F1-Weighted  :  0.6009

              precision    recall  f1-score   support

    negative     0.2459    0.6356    0.3546       118
     neutral     0.7602    0.6823    0.7191       576
    positive     0.6266    0.4092    0.4951       369

    accuracy                         0.5823      1063
   macro avg     0.5442    0.5757    0.5229      1063
weighted avg     0.6567    0.5823    0.6009      1063


  FINAL SUMMARY
  Accuracy     :  0.5823
  F1-Macro     :  0.5229
  F1-Weighted  :  0.6009
  Model saved  :  saved_models\bilstm_sentiment.pt

